In [1]:
#Load Project Environment
import Pkg
Pkg.activate(dirname(@__DIR__))
Pkg.instantiate()

  Activating project at `c:\Users\pbb62\Documents\Repositories\CHANCE_C.jl`


In [2]:
#Load Packages
using CSV, DataFrames
using DataStructures
using Agents
using Statistics,StatsBase,Distributions

include(joinpath(dirname(@__DIR__), "src/CHANCE_C.jl"))
using .CHANCE_C

In [3]:
###Load Input data:
##For flood history input
f_df = DataFrame(CSV.File(joinpath(dirname(@__DIR__), "data", "synth_flood_phil.csv")))

##For BG
#open bg file
phil_bg = DataFrame(CSV.File(joinpath(dirname(@__DIR__), "data/philly_bg_2019.csv")))
#groupby BG
grouped_phil_bg = groupby(phil_bg, :GEOID)

##load pop data
phil_cbsa_base_pop = DataFrame(CSV.File(joinpath(dirname(dirname(@__DIR__)), "philadelphia-data/census_data/synth_pop/pop_files/philly_cbsa_pop_0.csv")))
#drop missing values
dropmissing!(phil_cbsa_base_pop, :NP)

#Subset to Phil. County (Not part of function)
phil_base_pop = subset(phil_cbsa_base_pop, :county => x -> x .== 42101)

Row,serialno,year,state,puma,rep,county,tract,bg,puma10,GEOID,RAC1P,NP,HINCP,ADJINC,adj_income_2019
,String15,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64?,Float64,Float64?,Float64?,Float64?
1,2015000000403,2015,42,3201,0,42101,35100,1,4203201,421010351001,1.0,3.0,100000.0,1.08047,108047.0
2,2015000000403,2015,42,3201,0,42101,35200,1,4203201,421010352001,1.0,3.0,100000.0,1.08047,108047.0
3,2015000000403,2015,42,3201,0,42101,35500,3,4203201,421010355003,1.0,3.0,100000.0,1.08047,108047.0
4,2015000000403,2015,42,3201,0,42101,35500,3,4203201,421010355003,1.0,3.0,100000.0,1.08047,108047.0
5,2015000000403,2015,42,3201,0,42101,36100,1,4203201,421010361001,1.0,3.0,100000.0,1.08047,108047.0
6,2015000000403,2015,42,3201,0,42101,36201,3,4203201,421010362013,1.0,3.0,100000.0,1.08047,108047.0
7,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0
8,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0
9,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0


In [ ]:
names(grouped_phil_bg)

In [117]:
#Define input Parameters
no_of_years = 35
start_year = 1980
no_hhs_per_agent=10
grouped = true
group_col = "adj_income_2019"
cutoff_dict = OrderedDict("low"=> [-60000.00,25000.00], "medium"=>[25000.00,75000.00], "high"=>[75000.00, 1e7])
bg_cat = Dict(:col =>"income_cat", :group => ["low", "medium", "high"])
house_budget_mode = "perc"
house_choice_mode = "flood_mem_utility"
risk_averse = 0.3
flood_mem = 10
seed = 1500

1500

In [73]:
### Calculate Flood matrix and Dict for ABM input
f_dict, f_matrix = CHANCE_C.flood_history(f_df; no_of_years = no_of_years, start_year = start_year)

([0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;;], Dict(5 => (1, 5), 16 => (1, 16), 20 => (1, 20), 35 => (1, 35), 12 => (1, 12), 24 => (1, 24), 28 => (1, 28), 8 => (1, 8), 17 => (1, 17), 30 => (1, 30)…))

In [118]:
### Initialize ABM
phil_abm = CHANCE_C.Simulator(phil_bg, phil_base_pop, f_dict, f_matrix, CHANCE_C.model_step!; no_of_years = no_of_years, no_hhs_per_agent = no_hhs_per_agent,
house_budget_mode = house_budget_mode, house_choice_mode = house_choice_mode, grouped = grouped, group_col = group_col, cutoff_dict = cutoff_dict, bg_cat = bg_cat,
risk_averse = risk_averse, flood_mem = flood_mem, seed = seed)

StandardABM with 64063 agents of type Union{BlockGroup, HHAgent, Main.CHANCE_C.Queue}
 agents container: Dict
 space: GridSpace with size (37, 37), metric=chebyshev, periodic=true
 scheduler: Agents.Schedulers.ByType
 properties: df, total_population, flood_hazard, agent_creation, relo_sampler, agent_relocate, build_develop, house_price, hh_utilities_df, no_of_years, flood_matrix, flood_dict, tick

In [ ]:
length([a for a in allagents(phil_abm) if a isa HHAgent])

In [119]:
### Test model functions
test_bg = phil_abm[10]
println("Occupied: ",test_bg.occupied_units)
println("Available: ",test_bg.available_units)
println("Population: ",test_bg.population)

Occupied: Dict("medium" => 35, "high" => 28, "low" => 12)
Available: Dict("medium" => 2, "high" => 12, "low" => 0)
Population: 946


In [ ]:
length([a for a in agents_in_position(test_bg, phil_abm) if a isa HHAgent && a.group == "medium"])

In [120]:
CHANCE_C.agent_prob!(test_bg, phil_abm)

869

In [ ]:
println("Occupied: ",test_bg.occupied_units)
println("Available: ",test_bg.available_units)
println("Population: ",test_bg.population)

In [77]:
collect(agents_in_position(phil_abm[0].pos, phil_abm))

8-element Vector{AbstractAgent}:
 Main.CHANCE_C.Queue(0, (20, 11), :relocating)
 HHAgent(11710, (20, 11), 0, 9, "medium", 1.0, 1, 25732.353999999996, Dict(10 => 18988.181906421916), "perc", 0, 0.95, true, 34224.03081999999, 0.33)
 HHAgent(11712, (20, 11), 0, 10, "medium", 1.0, 1, 30717.706945, Dict(10 => 18988.181906421916), "perc", 0, 0.95, false, 40854.55023685, 0.33)
 HHAgent(11724, (20, 11), 0, 10, "medium", 1.0, 1, 45933.09666000001, Dict(10 => 18988.181906421916), "perc", 0, 0.95, true, 61091.01855780002, 0.33)
 HHAgent(11750, (20, 11), 0, 10, "high", 1.0, 1, 95826.23907500002, Dict(10 => 158976.05201352175), "perc", 0, 0.95, true, 127448.89796975003, 0.33)
 HHAgent(11769, (20, 11), 0, 10, "high", 1.0, 2, 322923.49792999995, Dict(10 => 158976.05201352175), "perc", 0, 0.95, true, 429488.25224689994, 0.33)
 HHAgent(11771, (20, 11), 0, 10, "high", 1.0, 1, 506291.51215, Dict(10 => 158976.05201352175), "perc", 0, 0.95, true, 673367.7111595001, 0.33)
 HHAgent(11772, (20, 11), 0, 4, "hi

In [105]:
function agent_locate(agent::CHANCE_C.Queue, model::ABM; levee = false, f_e = 0.0, bg_sample_size = 10, house_choice_mode = "simple_anova_utility",
    budget_reduction_perc = 0.10, penalty = 50, migrate_prob = 0.05)
    
    loc_df = copy(model.df)
    # Preallocate the DataFrame with a reasonable initial capacity
    #bg_sample = DataFrame(hh_id = Int64[], bg_id = Int64[], GEOID = Int64[], cat = String[], bg_utility = Float64[])

    # Use view or filter instead of multiple list comprehensions
    moving_agents = sort!([a for a in agents_in_position(agent, model) if a isa HHAgent], by=a -> a.income, rev=true)

    current_index = 1
    #Preallocate some vectors to reduce memory allocations
    hh_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
    bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
    bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

    for hh_agent in moving_agents
        # Consolidate budget selection logic
        bg_budget = if house_choice_mode == "simple_avoidance_utility"
            hh_agent.avoidance ? 
                subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
                subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true)
        elseif house_choice_mode == "budget_reduction"
            new_house_budget = hh_agent.house_budget * (1 - budget_reduction_perc)
            hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, hh_agent.house_budget)
            subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
        else
            subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
        end

        # Use a more efficient sampling approach
        try
            
            # Precompute weights to avoid repeated calculations
            weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
            
            # Check for available locations more efficiently
            valid_locations = findall(weights .> 0)
            if isempty(valid_locations)
                throw(ErrorException("No affordable locations with available units"))
            end

            #Sample from affordable locations based on weights
            sample_size = min(length(valid_locations), bg_sample_size)
            sampled_indices = sample(abmrng(model), valid_locations, sample_size, replace=false)
    
            #Grab utilities from sampled locations
            bg_sel = Iterators.filter(bg -> bg isa BlockGroup && bg.GEOID in bg_budget[sampled_indices, :GEOID], allagents(model)).id
            loc_utilities = getindex.(getproperty.(getindex.(Ref(model), bg_sel), :current_utility), bg_budget[sampled_indices, :income_cat])
            # Find indices of block groups with better utilities than current agent location
            current_utility = first(values(hh_agent.utility))
            opt_locs = findall(>(current_utility), loc_utilities)

            # Check if any moves are possible
            if isempty(opt_locs)
                throw(ErrorException("No better locations found"))
            end
            best_indices = sampled_indices[opt_locs]

            #Append future block group properties to vectors
            ind_length = length(best_indices)

            copyto!(hh_ids, current_index, fill(hh_agent.id, ind_length), 1, ind_length)
            copyto!(bg_ids, current_index, collect(bg_sel)[opt_locs], 1, ind_length)
            copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
            copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
            copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)

            current_index += ind_length

        catch
            # Migration logic remains similar
            if rand(abmrng(model), Binomial(1, migrate_prob)) == 1
                last_bg = model[first(keys(hh_agent.utility))]
                move_agent!(hh_agent, last_bg.pos, model)
                last_bg.occupied_units[hh_agent.group] += 1
                last_bg.available_units[hh_agent.group] -= 1
                last_bg.population += getproperty(hh_agent, :no_hhs_per_agent) * getproperty(hh_agent, :hh_size)
            else
                remove_agent!(hh_agent, model)
            end
        end
    end
    
    ##Create df from vectors, append to model properties df
    #Remove extra undef values by using current index
    bg_sample = DataFrame(hh_id = hh_ids[1:current_index-1], bg_id = bg_ids[1:current_index-1], 
    GEOID = bg_GEOID[1:current_index-1], cat = bg_cat[1:current_index-1], bg_utility = bg_utilities[1:current_index-1])
    
    append!(model.hh_utilities_df, bg_sample)
end

agent_locate (generic function with 1 method)

In [106]:
agent_locate(phil_abm[0], phil_abm)


Row,hh_id,bg_id,GEOID,cat,bg_utility
,Int64,Int64,Int64,String,Float64


In [93]:
phil_abm.hh_utilities_df

Row,hh_id,bg_id,GEOID,cat,bg_utility
,Int64,Int64,Int64,String,Float64
1,11772,465,421010142001,high,1.69437e5
2,11772,106,421010030022,high,1.97513e5
3,11772,539,421010008042,medium,2.77539e5
4,11769,541,421010353022,medium,197245.0
5,11769,1251,421010267007,medium,1.63954e5
6,11724,60,421010133002,low,2.13147e5
7,11724,434,421010268002,low,60260.3
8,11724,70,421010022003,low,28247.5
9,11712,60,421010018002,low,2.13147e5


In [ ]:
#Breakdown agent relocation function:
loc_df = copy(phil_abm.df)
#Create a GEOID-to-BlockGroup lookup
geoid_to_bg = Dict{Int64, Int64}()
for bg in allagents(phil_abm)
    if bg isa BlockGroup
        geoid_to_bg[bg.GEOID] = bg.id
    end
end
house_choice_mode = "simple_anova_utility"
bg_sample_size = 10
# Use view or filter instead of multiple list comprehensions
moving_agents = sort!([a for a in agents_in_position(phil_abm[0], phil_abm) if a isa CHANCE_C.HHAgent], by=a -> a.income, rev=true)
agent_sel = moving_agents[1]
current_index = 1
# Preallocate some vectors to reduce memory allocations
bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

# Consolidate budget selection logic
bg_budget = if house_choice_mode == "simple_avoidance_utility"
    agent_sel.avoidance ? 
        subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
        subset(loc_df, :market_value => n -> n .<= agent_sel.house_budget, skipmissing=true)
elseif house_choice_mode == "budget_reduction"
    new_house_budget = agent_sel.house_budget * (1 - budget_reduction_perc)
    hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, agent_sel.house_budget)
    subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
else
    subset(loc_df, :market_value => n -> n .<= agent_sel.house_budget, skipmissing=true, view = true)
end




In [126]:
# Use a more efficient sampling approach
#try
    # Precompute weights to avoid repeated calculations
weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
            
    # Check for available locations more efficiently
valid_locations = findall(weights .> 0)
if isempty(valid_locations)
    throw(ErrorException("No affordable locations with available units"))
end
    # Efficient sampling
sample_size = min(length(valid_locations), bg_sample_size)
sampled_indices = sample(abmrng(phil_abm), valid_locations, sample_size, replace=false)
    #bg_options = bg_budget[sampled_indices, :]
    
    #Grab utilities from sampled locations
#bg_sel = map(bg -> bg.id, Iterators.filter(bg -> bg isa BlockGroup && bg.GEOID in bg_budget[sampled_indices, :GEOID], allagents(phil_abm)))
loc_utilities = [phil_abm[geoid_to_bg[row.GEOID]].current_utility[row.income_cat] for row in eachrow(bg_budget[sampled_indices, [:GEOID, :income_cat]])]
# Find indices of block groups with better utilities than current agent location
current_utility = first(values(agent_sel.utility))
opt_locs = findall(>(current_utility), loc_utilities)
# Check if any moves are possible
if isempty(opt_locs)
    throw(ErrorException("No better locations found"))
end
best_indices = sampled_indices[opt_locs]
#catch
#    println("didnt work!")
#end

3-element Vector{Int64}:
  963
   19
 1639

In [127]:
println(sampled_indices)
println(opt_locs)
println(best_indices)


[1153, 1922, 259, 963, 1094, 1115, 1482, 19, 1945, 1639]
[4, 8, 10]
[963, 19, 1639]


In [133]:
getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID])

3-element Vector{Int64}:
   60
 1299
 1178

In [ ]:
ind_length = length(best_indices)   
#bg_ids = getproperty.(bg_sel[opt_locs], :id)
#bg_GEOID = bg_budget[sampled_indices, :GEOID]
#bg_cat = bg_budget[sampled_indices, :income_cat]
#bg_utilities = loc_utilities[opt_locs]

copyto!(bg_ids, current_index, getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID]), 1, ind_length)
copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)
current_index += ind_length


# Append to bg_sample
#append!(bg_sample, move_df)

2

In [69]:
fill(agent_sel.id, ind_length)

2-element Vector{Int64}:
 11772
 11772

In [ ]:
agent_locate(phil_abm[0], phil_abm)

In [ ]:
agent_relocate(phil_abm[0], phil_abm)

In [ ]:
sort!(filter(a -> a isa HHAgent, agents_in_position(phil_abm[0], phil_abm)), by=a -> a.income, rev=true)

In [ ]:
step!(phil_abm)

In [ ]:
phil_abm.tick